# 02 - Preprocessing, Imputation and Feature Scaling

Ce notebook met en œuvre la chaîne de prétraitement des données de la tuberculose (OMS) conformément aux décisions validées lors de l'Analyse Exploratoire des Données (EDA).

### Synthèse des décisions appliquées (D1 - D9) :
- **D1** : Exclure les bornes d'incertitude (`_lo`, `_hi`), les effectifs bruts (`_num`) et la variable redondante (`cfr_pct`). Conservation de `cfr`.
- **D4** : Application de la transformation logarithmique (`log1p`) sur les taux avant la standardisation pour gérer la forte asymétrie (*right skewness*).
- **D7** : Imputation temporelle par pays, puis par la médiane de la région OMS pour les valeurs résiduelles, accompagnée d'indicateurs binaires d'imputation (*imputation flags*).
- **D9** : Encodage disjonctif complet (*One-Hot Encoding*) de la région OMS (`g_whoregion`).

## 1. Chargement des données et bibliothèques

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Chargement du jeu de données brut
raw_data_path = "./02_data/raw_data/TB_burden_countries_2025-09-06.csv"
df = pd.read_csv(raw_data_path)

print(f"Dimensions initiales du dataset : {df.shape}")
df.head()

## 2. Sélection des variables d'intérêt (Décision D1)

In [ ]:
# Sélection des 13 variables cibles
features_to_keep = [
    'country', 'iso3', 'g_whoregion', 'year',
    'e_pop_num', 'e_inc_100k', 'e_mort_100k', 
    'e_mort_exc_tbhiv_100k', 'e_tbhiv_prct', 'e_inc_tbhiv_100k',
    'cfr', 'c_cdr', 'c_newinc_100k'
]

df_clean = df[features_to_keep].copy()
print(f"Dimensions après filtrage des colonnes : {df_clean.shape}")

## 3. Gestion des valeurs manquantes & Traçabilité (Décision D7)

In [ ]:
numeric_cols = [
    'e_pop_num', 'e_inc_100k', 'e_mort_100k', 
    'e_mort_exc_tbhiv_100k', 'e_tbhiv_prct', 
    'e_inc_tbhiv_100k', 'cfr', 'c_cdr', 'c_newinc_100k'
]

# Ajout des drapeaux d'imputation (imputation flags)
for col in numeric_cols:
    df_clean[f'{col}_is_imputed'] = df_clean[col].isna().astype(int)

# Étape A : Interpolation linéaire temporelle par pays
df_clean[numeric_cols] = df_clean.groupby('country')[numeric_cols].transform(
    lambda group: group.interpolate(method='linear', limit_direction='both')
)

# Étape B : Imputation par la médiane de la région OMS pour les valeurs manquantes résiduelles
df_clean[numeric_cols] = df_clean.groupby('g_whoregion')[numeric_cols].transform(
    lambda group: group.fillna(group.median())
)

print(f"Valeurs manquantes restantes : {df_clean[numeric_cols].isna().sum().sum()}")
# Export du dataset nettoyé à l'échelle brute
df_clean.to_csv("./02_data/processed_data/TB_burden_processed_raw_scale.csv", index=False)

## 4. Transformations pour la modélisation (Décisions D4 & D9)
- **Transformation Log1p** : Stabilisation des variances et atténuation de l'impact des valeurs extrêmes.
- **One-Hot Encoding** : Encodage de la région OMS (`g_whoregion`).
- **Standardisation** : Application du `StandardScaler` (moyenne=0, variance=1).

In [ ]:
# Transformation Logarithmique (log1p)
df_log = df_clean.copy()
df_log[numeric_cols] = np.log1p(df_log[numeric_cols])

# Encodage de la région OMS
df_encoded = pd.get_dummies(df_log, columns=['g_whoregion'], prefix='region', dtype=int)

# Standardisation des caractéristiques numériques
scaler = StandardScaler()
df_scaled = df_encoded.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

# Sauvegarde du fichier final prêt pour la PCA et le Clustering
output_path = "./02_data/processed_data/TB_burden_prepared_for_clustering.csv"
df_scaled.to_csv(output_path, index=False)
print(f"Préparation terminée. Fichier généré : {output_path}")
print(f"Dimensions finales : {df_scaled.shape}")